# Demo: traductor español ↔ quechua (Ayacucho)
NLLB-200-distilled-600M ajustado. Funciona en Colab con GPU T4 (Runtime → Change runtime type → T4) o en CPU.

**Antes de ejecutar:** pon en `PESOS` el repo de Hugging Face (o carpeta de Drive) con los pesos entrenados (ver README).

In [ ]:
!pip -q install transformers sentencepiece sacrebleu

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
PESOS = 'REEMPLAZAR/por-el-repo-de-pesos'   # p. ej. 'usuario/nllb600m-es-quy' o '/content/drive/MyDrive/mejor'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(PESOS)
modelo = AutoModelForSeq2SeqLM.from_pretrained(PESOS).to(dev).eval()
COD = {'es': 'spa_Latn', 'qu': 'quy_Latn'}

def traducir(texto, origen='es', destino='qu'):
    tok.src_lang = COD[origen]
    enc = tok(texto, return_tensors='pt', truncation=True, max_length=128).to(dev)
    with torch.inference_mode():
        gen = modelo.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(COD[destino]), num_beams=4, max_new_tokens=128)
    return tok.batch_decode(gen, skip_special_tokens=True)[0]


In [ ]:
for f in ['Buenos días, ¿cómo estás?', 'La comunidad se reunirá mañana para sembrar papa.']:
    print(f, '->', traducir(f, 'es', 'qu'))
print(traducir('Allin p’unchay', 'qu', 'es'))

## Reproducir la evaluación en test
Requiere `data/limpio/ayacucho/test.tsv` (se genera con `python src/limpiar_datos.py`, ver README).

In [ ]:
import pandas as pd
from sacrebleu.metrics import CHRF, BLEU
test = pd.read_csv('data/limpio/ayacucho/test.tsv', sep='	', keep_default_na=False).sample(200, random_state=0)
hip = [traducir(t, 'es', 'qu') for t in test['es']]
print('chrF++', CHRF(word_order=2).corpus_score(hip, [test['qu'].tolist()]).score, '| BLEU', BLEU().corpus_score(hip, [test['qu'].tolist()]).score)